In [1]:
# Fix 1: TensorFlow/protobuf crash in this environment ("MessageFactory has no attribute GetPrototype")
# This is typically resolved by forcing protobuf to use the pure-Python implementation BEFORE importing TF.
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import cv2

# Kaggle paths (keep as in original intention)
DATA_DIR = "../input/petfinder-pawpularity-score"
TRAIN_CSV_PATH = f"{DATA_DIR}/train.csv"
TEST_CSV_PATH = f"{DATA_DIR}/test.csv"
TRAIN_IMG_DIR = f"{DATA_DIR}/train"
TEST_IMG_DIR = f"{DATA_DIR}/test"

IMG_SIZE = (224, 224)


def load_images_by_id(ids, img_dir, img_size=IMG_SIZE):
    """Load images in the same order as ids to keep correct alignment with tabular rows."""
    imgs = np.empty((len(ids), img_size[1], img_size[0], 3), dtype=np.uint8)
    for idx, _id in enumerate(ids):
        p = f"{img_dir}/{_id}.jpg"
        im = cv2.imread(p)
        if im is None:
            raise FileNotFoundError(f"Could not read image: {p}")
        im = cv2.resize(im, img_size, interpolation=cv2.INTER_AREA)
        imgs[idx] = im
    return imgs


train_csv = pd.read_csv(TRAIN_CSV_PATH)

# Keep the original feature logic (12 metadata columns). Handle the space in "Subject Focus".
feature_cols = [c for c in train_csv.columns if c not in ["Id", "Pawpularity"]]
train_data = train_csv[feature_cols].to_numpy(dtype=np.float32)
train_label = train_csv["Pawpularity"].to_numpy(dtype=np.float32)

# Load train images aligned to train_csv order (prevents label/image mismatch from glob ordering)
train_imgs = load_images_by_id(train_csv["Id"].tolist(), TRAIN_IMG_DIR).astype(
    np.float32
)

# Optional: scale images to [0,1] (doesn't change architecture/training loop; improves stability)
train_imgs /= 255.0

print("train_data:", train_data.shape, train_data.dtype)
print("train_imgs:", train_imgs.shape, train_imgs.dtype)
print("train_label:", train_label.shape, train_label.dtype)



2026-01-23 16:03:55.450457: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769184235.463352      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769184235.467406      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-23 16:03:55.482569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

train_data: (8920, 12) float32
train_imgs: (8920, 224, 224, 3) float32
train_label: (8920,) float32


In [2]:
# Fix 2: Keras Input shape must be a tuple; shape=(12) is invalid, use shape=(12,)
inputA = keras.Input(shape=(12,), name="meta_input")
inputB = keras.Input(shape=(224, 224, 3), name="image_input")

x = layers.Dense(12, activation="softmax")(inputA)
x = keras.Model(inputs=inputA, outputs=x)

y = layers.Conv2D(96, (11, 11), padding="same", activation="relu", strides=4)(inputB)
y = layers.MaxPooling2D(pool_size=(3, 3), strides=2)(y)
y = layers.Conv2D(256, (5, 5), padding="same", activation="relu", strides=2)(y)
y = layers.MaxPooling2D(pool_size=(3, 3), strides=2)(y)
y = layers.Conv2D(384, (3, 3), padding="same", activation="relu")(y)
y = layers.Conv2D(384, (3, 3), padding="same", activation="relu")(y)
y = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(y)
y = layers.MaxPooling2D(pool_size=(3, 3), strides=2)(y)
y = layers.Flatten()(y)
y = layers.Dense(12, activation="softmax")(y)
y = keras.Model(inputs=inputB, outputs=y)

combined = layers.Concatenate()([x.output, y.output])
z = layers.Dense(24, activation="relu")(combined)
z = layers.Dense(1, activation="linear")(z)

model = keras.Model([x.input, y.input], z)

# Avoid plot_model failures if graphviz/pydot aren't available; not needed for training/submission
try:
    keras.utils.plot_model(model, show_shapes=True)
except Exception as e:
    print("plot_model skipped:", repr(e))

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[tf.keras.metrics.RootMeanSquaredError()],
)

# Keep original split counts and training approach
model.fit(
    [train_data[:7930], train_imgs[:7930]], train_label[:7930], epochs=25, verbose=2
)
results = model.evaluate(
    [train_data[7930:], train_imgs[7930:]], train_label[7930:], verbose=0
)
print("eval:", dict(zip(model.metrics_names, results)))



2026-01-23 16:04:50.618148: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1769184290.618547      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c5:00.0, compute capability: 8.6


Epoch 1/25


I0000 00:00:1769184300.871928      65 service.cc:148] XLA service 0x7f320c013320 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769184300.871953      65 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-23 16:05:00.904517: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769184301.081332      65 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1769184302.735894      65 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


248/248 - 8s - 31ms/step - loss: 1752.5093 - root_mean_squared_error: 41.8630
Epoch 2/25
248/248 - 2s - 8ms/step - loss: 1280.7655 - root_mean_squared_error: 35.7878
Epoch 3/25
248/248 - 2s - 8ms/step - loss: 758.1221 - root_mean_squared_error: 27.5340
Epoch 4/25
248/248 - 2s - 8ms/step - loss: 497.4220 - root_mean_squared_error: 22.3030
Epoch 5/25
248/248 - 2s - 8ms/step - loss: 435.2179 - root_mean_squared_error: 20.8619
Epoch 6/25
248/248 - 2s - 8ms/step - loss: 428.5670 - root_mean_squared_error: 20.7019
Epoch 7/25
248/248 - 2s - 8ms/step - loss: 428.2420 - root_mean_squared_error: 20.6940
Epoch 8/25
248/248 - 2s - 8ms/step - loss: 428.2247 - root_mean_squared_error: 20.6936
Epoch 9/25
248/248 - 2s - 8ms/step - loss: 428.2250 - root_mean_squared_error: 20.6936
Epoch 10/25
248/248 - 2s - 8ms/step - loss: 428.1964 - root_mean_squared_error: 20.6929
Epoch 11/25
248/248 - 2s - 8ms/step - loss: 428.0360 - root_mean_squared_error: 20.6890
Epoch 12/25
248/248 - 2s - 8ms/step - loss: 428.1

In [3]:
test_csv = pd.read_csv(TEST_CSV_PATH)
test_data = test_csv[feature_cols].to_numpy(dtype=np.float32)

# Fix 3: Load test images aligned with test_csv Id order (glob order can mismatch)
test_imgs = load_images_by_id(test_csv["Id"].tolist(), TEST_IMG_DIR).astype(np.float32)
test_imgs /= 255.0

prediction = model.predict([test_data, test_imgs], batch_size=32, verbose=1).reshape(-1)

# Create submission in required format
prediction_df = pd.DataFrame({"Id": test_csv["Id"].values, "Pawpularity": prediction})

# (Optional, score-neutral safety) Clip to plausible range [0, 100]
prediction_df["Pawpularity"] = prediction_df["Pawpularity"].clip(0, 100)

prediction_df.to_csv("submission.csv", header=True, index=False)
print(prediction_df.head())
print("Wrote submission.csv with shape:", prediction_df.shape)

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
                                 Id  Pawpularity
0  ee51b99832f1ba868f646df93d2b6b81    38.397533
1  caddfb3f8bff9c4b95dbe022018eea21    38.435432
2  582eeabd4a448a53ebb79995888a4b0b    38.101681
3  afc1ad7f0c5eea880759d09e77f7deee    38.397533
4  d5bdf3446e86ce4ec67ce7a00f1cccc2    38.307556
Wrote submission.csv with shape: (992, 2)
